# 🤖 ANA 699 Robotics Capstone — Daniel Kast
## HalfCheetah EDA + Decision Transformer
**Environment:** M1 iMac (Apple Silicon baseline)  
**Dataset:** D4RL HalfCheetah Medium-v2 + Medium-Expert-v2 combined  
**Objective:** Perform exploratory data analysis then train a Decision Transformer

---
### 📋 Notebook Structure
1. Imports & Setup
2. Dataset Download
3. Data Loading & Inspection
4. Exploratory Data Analysis (EDA)
5. Decision Transformer Architecture
6. Training
7. Evaluation & Results



In [1]:
# ============================================================
# ANA 699 Robotics Capstone — Daniel Kast
# EDA + Decision Transformer — HalfCheetah Medium + Expert
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import h5py
import os
from huggingface_hub import hf_hub_download

# Plot settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("✅ All imports successful!")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")

Matplotlib is building the font cache; this may take a moment.


✅ All imports successful!
NumPy: 2.2.6
Pandas: 2.3.3


### ⚠️ Setup Notes
- D4RL pip install fails on M1 due to `pybullet` incompatibility with Apple Silicon
- **Solution:** Load datasets directly as HDF5 files from Berkeley RAIL public mirror
- `mujoco_py` is deprecated — using `mujoco 3.6.0` instead
- gym replaced with `gymnasium`

In [4]:
# ============================================================
# Cell 2 — Download HalfCheetah Datasets directly
# ============================================================
import urllib.request

# Create data directory inside your experiment folder
os.makedirs('data', exist_ok=True)

# Direct download URLs from D4RL public mirror
urls = {
    'halfcheetah_medium-v2.hdf5': 'http://rail.eecs.berkeley.edu/datasets/offline_rl/gym_mujoco_v2/halfcheetah_medium-v2.hdf5',
    'halfcheetah_medium_expert-v2.hdf5': 'http://rail.eecs.berkeley.edu/datasets/offline_rl/gym_mujoco_v2/halfcheetah_medium_expert-v2.hdf5'
}

for filename, url in urls.items():
    filepath = f'data/{filename}'
    if not os.path.exists(filepath):
        print(f"📥 Downloading {filename}...")
        urllib.request.urlretrieve(url, filepath)
        print(f"✅ Saved to {filepath}")
    else:
        print(f"✅ Already exists: {filepath}")

medium_path = 'data/halfcheetah_medium-v2.hdf5'
expert_path = 'data/halfcheetah_medium_expert-v2.hdf5'
print("\n🎉 Both datasets ready!")

📥 Downloading halfcheetah_medium-v2.hdf5...
✅ Saved to data/halfcheetah_medium-v2.hdf5
📥 Downloading halfcheetah_medium_expert-v2.hdf5...
✅ Saved to data/halfcheetah_medium_expert-v2.hdf5

🎉 Both datasets ready!


## 📥 Section 2 — Dataset Download

We are loading two D4RL HalfCheetah datasets directly from the Berkeley RAIL public mirror as HDF5 files.

### ⚠️ Issues Encountered
| Issue | Detail | Resolution |
|---|---|---|
| `pybullet` build failure | D4RL pip install fails on M1 Apple Silicon — pybullet cannot compile on ARM architecture | Uninstalled D4RL and gym, switched to HDF5 direct download |
| `mujoco_py` not installed | Deprecated library, incompatible with modern MuJoCo 3.x | Not needed — using `mujoco 3.6.0` directly |
| gym version conflict | D4RL requires `gym<0.24.0` but installed version was `0.26.2` | Uninstalled gym entirely, switched to `gymnasium` |
| HuggingFace repo `offline-rl-datasets/d4rl-mujoco-halfcheetah` | Repository not found — 401 error | Tried alternate repo |
| HuggingFace repo `kzl/d4rl` | Repository is private — 401 Unauthorized | Switched to Berkeley RAIL direct download |

### ✅ Final Solution
Downloaded HDF5 files directly from Berkeley RAIL public mirror:
- `http://rail.eecs.berkeley.edu/datasets/offline_rl/gym_mujoco_v2/`
- No authentication required
- Same official D4RL data

### Datasets
| Dataset | Description | Expected Size |
|---|---|---|
| `halfcheetah_medium-v2` | Trajectories from a medium-quality policy | ~1M steps |
| `halfcheetah_medium_expert-v2` | Mix of medium + expert policy trajectories | ~2M steps |

### Why combine medium + expert?
Combining both datasets gives our Decision Transformer exposure to a wide range of behavior quality — from average performance to near-optimal. This is key for **Return-to-Go (RTG) conditioning** — the model learns to associate high RTG prompts with expert-like actions.

In [6]:
# ============================================================
# Cell 3 — Load and Inspect Both Datasets (fixed)
# ============================================================

def load_dataset(filepath):
    """Load a D4RL HDF5 dataset into a dictionary of numpy arrays"""
    data = {}
    with h5py.File(filepath, 'r') as f:
        print(f"\n📂 File: {filepath}")
        print(f"   Keys: {list(f.keys())}")
        for key in f.keys():
            item = f[key]
            # Only load datasets, skip groups
            if isinstance(item, h5py.Dataset):
                data[key] = item[()]
                print(f"   ✅ {key}: shape={data[key].shape}, dtype={data[key].dtype}")
            else:
                print(f"   ⏭️  {key}: (group, skipping)")
    return data

# Load both datasets
print("="*60)
print("MEDIUM DATASET")
print("="*60)
medium = load_dataset(medium_path)

print("\n" + "="*60)
print("MEDIUM-EXPERT DATASET")
print("="*60)
expert = load_dataset(expert_path)

MEDIUM DATASET

📂 File: data/halfcheetah_medium-v2.hdf5
   Keys: ['actions', 'infos', 'metadata', 'next_observations', 'observations', 'rewards', 'terminals', 'timeouts']
   ✅ actions: shape=(1000000, 6), dtype=float32
   ⏭️  infos: (group, skipping)
   ⏭️  metadata: (group, skipping)
   ✅ next_observations: shape=(1000000, 17), dtype=float32
   ✅ observations: shape=(1000000, 17), dtype=float32
   ✅ rewards: shape=(1000000,), dtype=float32
   ✅ terminals: shape=(1000000,), dtype=bool
   ✅ timeouts: shape=(1000000,), dtype=bool

MEDIUM-EXPERT DATASET

📂 File: data/halfcheetah_medium_expert-v2.hdf5
   Keys: ['actions', 'infos', 'next_observations', 'observations', 'rewards', 'terminals', 'timeouts']
   ✅ actions: shape=(2000000, 6), dtype=float32
   ⏭️  infos: (group, skipping)
   ✅ next_observations: shape=(2000000, 17), dtype=float32
   ✅ observations: shape=(2000000, 17), dtype=float32
   ✅ rewards: shape=(2000000,), dtype=float32
   ✅ terminals: shape=(2000000,), dtype=bool
   ✅ tim

## 🔍 Section 3 — Data Loading & Inspection

Now that both HDF5 files are downloaded locally we load them into memory as numpy arrays for inspection.

### ⚠️ Issues Encountered
| Issue | Detail | Resolution |
|---|---|---|
| `TypeError: Accessing a group is done with bytes or str, not <class 'slice'>` | Some HDF5 keys are Groups not Datasets — using `f[key][:]` fails on groups | Updated loader to check `isinstance(item, h5py.Dataset)` and skip groups |

### ✅ Keys Loaded
| Key | Type | Description |
|---|---|---|
| `observations` | Dataset | 17-dim state vector (joint positions + velocities) |
| `actions` | Dataset | 6-dim action vector (motor torques) |
| `rewards` | Dataset | Scalar reward per timestep |
| `next_observations` | Dataset | Next state after action |
| `terminals` | Dataset | Boolean — True at episode end |
| `infos` | Group | Skipped — metadata group, not needed |
| `metadata` | Group | Skipped — metadata group, not needed |

### Expected Output
- Medium: 1,000,000 timesteps
- Medium-Expert: 2,000,000 timesteps
- Total combined: 3,000,000 timesteps